### RF Signal Generator pulsing to test instrument

In [ ]:
#
#
#  Initialize the PulseBlaster and the Anritsu signal generator
#
#

from spinapi import *

# Enable the log file
pb_set_debug(0)

print("Copyright (c) 2015 SpinCore Technologies, Inc.");
print("Using SpinAPI Library version %s" % pb_get_version())
print("Found %d boards in the system.\n" % pb_count_boards())
 
pb_select_board(0)

if pb_init() != 0:
	print("Error initializing board: %s" % pb_get_error())
	input("Please press a key to continue.")
	exit(-1)

# Configure the core clock
pb_core_clock(500)

import pyvisa
rm = pyvisa.ResourceManager()
siggen = rm.open_resource('ASRL4::INSTR')
siggen.read_termination = '\r\n'
# siggen.write_termination = '\n'
siggen.baud_rate = 9600

In [ ]:
#-------------------------------------------------------------------------------
# Set signal generator frequency and output level and turn on 
#
siggen.write('FREQ 20.0 MHz')
mwpower = -35
siggen.write(f'OLVL {mwpower} DBM')
siggen.query('OLVL?')
siggen.write('LVL ON')

In [ ]:
siggen.write('LVL OFF')

Pulse the RF signal generator output

In [ ]:
#---------------------------------------------------------------------------
#
# Write a pulse program 
#
# Define binary strings for each output channel 
#   Note : here we use the first four channels (read right to left) for instrument control
#          and repeat those on channels 5-8 to monitor on an oscilloscope
#

Ton = 1*us
Toff = 1*us
#---------------------------------------------------------------------------
#
# Program the updated pulse program 
#
# Define binary strings for each output channel 
#   Note : here we use the first four channels (read right to left) for instrument control
#          and repeat those on channels 5-8 to monitor on an oscilloscope
#
bit0      = 0b00010001
bit1      = 0b00100010
bit2      = 0b01000100
bit3      = 0b10001000
zeros     = 0b00000000

# Initiate a pulse program on the PulseBlaster
pb_start_programming(PULSE_PROGRAM)

# reference high
start = pb_inst_pbonly(bit0 + bit2, Inst.CONTINUE, 0, Ton)

pb_inst_pbonly(zeros, Inst.BRANCH, start, Toff)

pb_stop_programming()
#---------------------------------------------------------------------------

# Trigger the board
pb_reset() 
pb_start()

print("Pressing a key will stop pulsing\n");
input("Please press a key to stop.")

pb_stop()
# pb_close()  # don't call close if you want to go again...

In [ ]:
#
# Stop the pulse generation and
#    turn off the generator output when done
#
pb_stop()
siggen.write('LVL OFF')

### Template for basic PulseBlaster programming

In [ ]:
#
#
#  Write some comments ...
#
#
from spinapi import *

# Enable the log file
pb_set_debug(0)

print("Copyright (c) 2015 SpinCore Technologies, Inc.");
print("Using SpinAPI Library version %s" % pb_get_version())
print("Found %d boards in the system.\n" % pb_count_boards())
 
pb_select_board(0)

if pb_init() != 0:
	print("Error initializing board: %s" % pb_get_error())
	input("Please press a key to continue.")
	exit(-1)

# Configure the core clock
pb_core_clock(500)

#Initialize useful variables
Tlaser = 2.5*ms
Tdelay = 4.5*ms
Tref = 10*ms
Twait = Tref - Tlaser - Tdelay


#---------------------------------------------------------------------------
#
# Write a pulse program 
#
# Define binary strings for each output channel 
#   Note : here we use the first four channels (read right to left) for instrument control
#          and repeat those on channels 5-8 to monitor on an oscilloscope
#
bit0      = 0b00010001
bit1      = 0b00100010
bit2      = 0b01000100
bit3      = 0b10001000
zeros     = 0b00000000

#Program the pulse program -------------------------------------------------
pb_start_programming(PULSE_PROGRAM)

# set first 4 bits to 1
start = pb_inst_pbonly(bit0 + bit1 + bit2 + bit3, Inst.CONTINUE, 0, Tlaser)

# turn off bits 0 and 2
pb_inst_pbonly(bit1 + bit2, Inst.CONTINUE, 0, Tdelay)

# set all to zero
pb_inst_pbonly(zeros, Inst.BRANCH, start, Twait)

pb_stop_programming()
#---------------------------------------------------------------------------

# Trigger the board
pb_reset() 

pb_start()

print("Pressing a key will stop pulsing\n");
input("Please press a key to stop.")

pb_stop()
pb_close()